In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA01 - Travel, Lodging, and Entertainment
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Coupa Data: 7 filtered files (ritm16070584_...)
   4. Finance Data: 6 files (f25_finance_data_...)
   
   BUSINESS LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + CANADA flag) will appear.
   2. SOURCE MERGE: Combines all 7 Coupa files and 6 Finance files into a single 
      unified transaction pipeline.
   3. STRING PARSING: Extracts the Cost Center and Category Code from Coupa's 
      hyphenated 'Account' string using split().
   4. TARGET CATEGORY CODES: Filters strictly for the ~30 approved target codes 
      provided in the master list.
   5. THE "793" EXCEPTION RULE: If a Category Code is 793, it is strictly locked to 
      Assessable Unit '101016'. If it maps to any other AU, it is intentionally dropped.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapping AS (
    -- 2. Pull the mapped Cost Centers from our finalized bootstrap view
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Combined_Coupa_Raw AS (
    -- 3a. Stack all 7 Coupa target files together
    SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- 3b. Extract the CC (1st block) and Category Code (3rd block) from Coupa's hyphenated string
    SELECT 
        Account AS Raw_String,
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

Combined_Finance_Raw AS (
    -- 4a. Stack all 6 Finance target files together
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Finance_Parsed AS (
    -- 4b. Standardize Finance columns to match Coupa format
    SELECT 
        CostCenter AS Raw_String,
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRY_CAST(TRIM(CatCode) AS INT) AS CatCode_Int
    FROM Combined_Finance_Raw
),

All_Transactions AS (
    -- 5. Merge the parsed Coupa and Finance streams into one master transaction list
    SELECT Raw_String, Cleaned_CC, CatCode_Int FROM Coupa_Parsed
    UNION ALL
    SELECT Raw_String, Cleaned_CC, CatCode_Int FROM Finance_Parsed
),

Flagged_Engagements AS (
    -- 6. Apply the ~30 Category Code filters and the strict 793 exception rule
    SELECT 
        t.Raw_String,
        t.Cleaned_CC,
        m.AU_ID
    FROM All_Transactions t
    INNER JOIN CC_Mapping m ON t.Cleaned_CC = m.Mapped_CC
    WHERE 
        -- RULE 1: Must be in the approved master category list
        t.CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892)
        
        -- RULE 2: If the code is 793, it MUST be mapped to AU 101016 to be counted
        AND NOT (t.CatCode_Int = 793 AND m.AU_ID != '101016')
),

Engagements_By_AU AS (
    -- 7. Count the total valid flagged transactions per AU
    SELECT 
        AU_ID,
        COUNT(Raw_String) AS Match_Count
    FROM Flagged_Engagements
    GROUP BY AU_ID
)

-- 8. Final Output: Strictly structured per Master Anchor
SELECT 
    mast.BusinessID, 
    mast.AU_Name, 
    mast.Business_Segment,
    'EBA01' AS QuestionID,               
    CASE 
        WHEN COALESCE(e.Match_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN Engagements_By_AU e 
    ON mast.BusinessID = e.AU_ID
ORDER BY mast.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA01 - Travel, Lodging, and Entertainment Traceability
   
   PURPOSE: 
   Isolates the data flow for EBA01 to show exactly how Coupa and Finance records 
   are parsed, whether their Category Codes trigger a flag, how they map to an AU, 
   and whether they get blocked by the strict "793" exception rule.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data for bridge verification
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Combined_Coupa_Raw AS (
    -- 2a. Stack all 7 Coupa target files together
    SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- 2b. Extract the CC and Category Code, tagging the source system
    SELECT 
        Account AS Raw_String,
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int,
        'Coupa' AS Source_System
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

Combined_Finance_Raw AS (
    -- 3a. Stack all 6 Finance target files together
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Finance_Parsed AS (
    -- 3b. Standardize Finance columns and tag the source system
    SELECT 
        CostCenter AS Raw_String,
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRY_CAST(TRIM(CatCode) AS INT) AS CatCode_Int,
        'Finance' AS Source_System
    FROM Combined_Finance_Raw
),

All_Transactions AS (
    -- 4. Merge the parsed Coupa and Finance streams
    SELECT Raw_String, Cleaned_CC, CatCode_Int, Source_System FROM Coupa_Parsed
    UNION ALL
    SELECT Raw_String, Cleaned_CC, CatCode_Int, Source_System FROM Finance_Parsed
),

CC_Mapping AS (
    -- 5. Pull the mapped Cost Centers from our finalized bootstrap view
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
)

-- 6. Output the explicit tracing details for every transaction
SELECT 
    t.Cleaned_CC AS Parsed_Cost_Center,
    m.AU_ID AS Mapped_AU_ID,
    mast.Master_AU_Name,
    t.Source_System,
    t.Raw_String AS Original_Record,
    t.CatCode_Int AS Parsed_Category_Code,
    
    -- Flag if the Category Code exists in our 30-item master list
    CASE WHEN t.CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892) 
         THEN 'YES' ELSE 'NO' END AS Is_Target_Category,
         
    
    -- Trace exactly how the 793 rule and standard rules are applied
    CASE 
        WHEN t.CatCode_Int = 793 AND m.AU_ID != '101016' THEN '❌ BLOCKED: 793 only valid for 101016'
        WHEN t.CatCode_Int = 793 AND m.AU_ID = '101016' THEN '✅ KEPT: Valid 793 mapping'
        WHEN t.CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 890, 892) THEN '✅ KEPT: Standard Category'
        ELSE '❌ DROPPED: Not a target category'
    END AS Pipeline_Status,
    
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'FAILED BRIDGE' ELSE 'SUCCESS' END AS Master_List_Status
    
FROM All_Transactions t
LEFT JOIN CC_Mapping m 
    ON t.Cleaned_CC = m.Mapped_CC
LEFT JOIN Master_AUs mast 
    ON m.AU_ID = mast.Master_Numeric_ID

-- Optional: Uncomment below to view only the flagged transactions
-- WHERE t.CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892)
ORDER BY t.Cleaned_CC;